# Homework 2. API requests

In [17]:
import requests
import re
import pandas as pd
import numpy as np
import json

#### Implement function get_<db_name> for each database that accepts an ID and outputs API response

In [2]:
def http_function(endpoint, **headers):
    response = requests.get(endpoint, headers=headers)
    return response

##### UniProt

In [3]:
def get_uniprot(accession):
  '''
  define request to get the data from Uniprot API
  '''
  endpoint = f"https://rest.uniprot.org/uniprotkb/{accession}.json"
  headers = {}

  return http_function(endpoint, **headers)

##### Testing for UniProt

In [4]:
get_uniprot('P11473')

<Response [200]>

In [5]:
get_uniprot('helloworld')

<Response [400]>

In [6]:
get_uniprot('helloworld').json()

{'url': 'http://rest.uniprot.org/uniprotkb/helloworld',
 'messages': ["The 'accession' value has invalid format. It should be a valid UniProtKB accession"]}

##### ENSEMBL

In [7]:
def get_ensembl(id):
    '''
    define request to get the data from Ensembl API
    '''
    endpoint = f"https://rest.ensembl.org/lookup/id/{id}"
    headers = {"Content-Type": "application/json"}

    return http_function(endpoint, **headers)

##### Testing for ENSEMBL

In [8]:
get_ensembl('ENSMUSG00000041147')

<Response [200]>

In [9]:
get_ensembl('helloworld')

<Response [400]>

In [10]:
get_ensembl('helloworld').json()

{'error': "ID 'helloworld' not found"}

#### Implement function parse_response_<db_name> for each database that accepts API response and outputs parsed information about the given ID

#### UniProt

In [11]:
def uniprot_parse_response(resp: dict):
    '''
    parse response from Uniprot and output
    organism, geneInfo, sequenceInfo, type
    '''
    if "messages" in resp:
        return f'Error in response: {resp['messages']}'
    try:
        output = {}
        output['organism'] = resp.get("organism")['scientificName']
        output['geneInfo'] = resp.get("genes")
        output['sequenceInfo'] = resp.get("sequence")
        output['type'] = 'protein'
    except Exception as e:
        return f'Error in python: {str(e)}'
    output = {resp.get("primaryAccession"): output}
    return output

#### Testing for UniProt

In [12]:
uniprot_parse_response(get_uniprot('P11473').json())

{'P11473': {'organism': 'Homo sapiens',
  'geneInfo': [{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312',
       'source': 'HGNC',
       'id': 'HGNC:12679'}],
     'value': 'VDR'},
    'synonyms': [{'value': 'NR1I1'}]}],
  'sequenceInfo': {'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS',
   'length': 427,
   'molWeight': 48289,
   'crc64': 'F95F300D042C4CB7',
   'md5': '0D963ACD4A34674368324EE026023597'},
  'type': 'protein'}}

#### ENSEMBLE

In [13]:
def ensembl_parse_response(resp: dict):
  '''
  parse Ensembl response and output
  object_type, assembly_name, species, db_type, biotype, display_name, id, description, canonical_transcript, source
  '''
  if "error" in resp:
    return f'Error in response: {resp['messages']}'
  try:
      output = {}
      output_keys = ['object_type', 'assembly_name', 'species', 'db_type', 'biotype', 'display_name', 'id', 
                     'description', 'canonical_transcript', 'source']
      for key in output_keys:
         output[key] = resp.get(key)
  except Exception as e:
      return f'Error in python: {str(e)}'
  output = {resp.get("id"): output}
  return output

#### Testing for ENSEMBLE

In [14]:
ensembl_parse_response(get_ensembl('ENSMUSG00000041147').json())

{'ENSMUSG00000041147': {'object_type': 'Gene',
  'assembly_name': 'GRCm39',
  'species': 'mus_musculus',
  'db_type': 'core',
  'biotype': 'protein_coding',
  'display_name': 'Brca2',
  'id': 'ENSMUSG00000041147',
  'description': 'breast cancer 2, early onset [Source:MGI Symbol;Acc:MGI:109337]',
  'canonical_transcript': 'ENSMUST00000044620.11',
  'source': 'ensembl_havana'}}

#### Implement function main that accepts a list of IDs and outputs a DataFrame with responses. Here you should define regular expressions to distinguish between IDs of different databases. Also, you should use here get and parse functions written beforehand

In [15]:
def main(ids: list):
  '''
  Function that iterates over all the provided IDs and parses them into dict,
  transforms into pandas.DataFrame, and return it
  {ID : info from parse_response(), ...}

  If ID is incorrect, it should return {ID : error message}
  '''
  
  uniprot_pattern = r"^[A-Z][0-9][A-Z0-9]{3}[0-9]$"
  ensembl_pattern = r"^[A-Z]+[A-Z]+[0-9]{11}$"
  output = {}
  
  for id_ in ids:
    try:
      if re.match(uniprot_pattern, id_):
        resp = uniprot_parse_response(get_uniprot(id_).json())
        output[id_] = resp
      elif re.match(ensembl_pattern, id_):
        resp = ensembl_parse_response(get_ensembl(id_).json())
        output[id_] = resp
      else:
        output[id_] = 'error:unknown database'
    except Exception as e:
      return f'error: {str(e)}'
  
  rows = [{'id': id_, 'info': info} for id_, info in output.items()]
  output = pd.DataFrame(rows)

  return output

In [16]:
main(['P11473', 'Q91XI3', 'hello', 'ENSG00000157764', 'ENSG00000139618'])

,id,info
0,P11473,"{'P11473': {'organism': 'Homo sapiens', 'geneI..."
1,Q91XI3,{'Q91XI3': {'organism': 'Ictidomys tridecemlin...
2,hello,error:unknown database
3,ENSG00000157764,"{'ENSG00000157764': {'object_type': 'Gene', 'a..."
4,ENSG00000139618,"{'ENSG00000139618': {'object_type': 'Gene', 'a..."
